# L5 Cleanup Sanity Check (No Clustering)

Purpose:
- Pull products in two target groups by `PROPOSED_LABEL`
- Run LLM sanity check from product name
- Output `KEEP`, `REASSIGN`, or `REVIEW`
- Optionally export one CSV per group

Rules:
- Exclude rows where `DESCRIPTION` contains `INSERT`
- Keep only the configured global category list as valid targets

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import get_bedrock_client, get_snowflake_session  # noqa: E402


In [2]:
# ===========================
# STEP 1: Config
# ===========================
DB = "SNOWFLAKE_LEARNING_DB"
SCHEMA = "SMCMAHON_PRODUCTS"
LABELS_TABLE = f"{DB}.{SCHEMA}.PROPOSED_L5"
PRODUCTS_TABLE = f"{DB}.{SCHEMA}.PRODUCTS_LCG"
JOIN_ID_COL = "PRODUCT_ID"

EXCLUDE_INSERT_PRODUCTS = True
ONLY_PRICED = False

TARGET_GROUPS = {
    "group_1_general_supplies": [
        "General Lab Supplies",
        "Lab Consumables",
    ],
    "group_2_equipment_instruments": [
        "Lab Equipment",
        "Lab Instruments",
        "General Lab Equipment",
        "Lab Instruments and Tools",
    ],
}

GLOBAL_CATEGORY_LIST = [
    "Hazardous Materials",
    "Shipping and Handling",
    "Safety and Compliance",
    "Chemical Storage",
    "Gloves and PPE",
    "Protective Wear",
    "First Aid Supplies",
    "Cleaning and Sterilization",
    "Labeling and Tape",
    "Lab Accessories",
    "Pipettes and Tips",
    "Tubing and Hoses",
    "Microplates",
    "Media and Agar",
    "Microscopy Supplies",
    "Laboratory Bottles",
    "Caps and Closures",
    "Laboratory Glassware",
    "Tubes and Vials",
    "Pyrex Glassware",
    "Chemical Safety",
    "Chromatography Columns",
    "Syringes and Needles",
    "Medical Accessories",
    "Chemical Reagents",
    "Solutions and Solvents",
    "Chemicals and Reagents",
    "Organic Compounds",
    "Amino Acids",
    "Inorganic Compounds",
    "Biological Reagents",
    "Molecular Biology Kits",
    "Antibodies",
    "Filtration Products",
    "Filter Papers",
    "Filtration Systems",
    "Filter Papers and Membranes",
    "Laboratory Hardware",
    "Analytical Balances",
    "Laboratory Equipment", 
    "Laboratory Instruments",
    "Laboratory Furniture",
    "Lab Seating",
    "Storage and Organization",
    "Lab Storage Solutions",
]

AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"

# Fast-path: use one proven working model for this run.
LLM_MODEL_IDS = [
    "anthropic.claude-3-haiku-20240307-v1:0",
]
LLM_TEMPERATURE = 0
SANITY_BATCH_SIZE = 50
LLM_PARSE_RETRIES = 1
ENABLE_MODEL_DISCOVERY = False
ALLOW_KEEP_DECISION = True
SANITY_GOAL_CONTEXT = (
    "We are cleaning up a product catalog for scientific and laboratory products and consumables. "
    "Our goal is to limit products that have fallen into general categories and place them elsewhere when logical, "
    "like cleaning out a junk drawer. If they cannot be confidently recategorized, return REVIEW."
)

SAVE_OUTPUT_FILES = True
OUT_DIR = PROJECT_ROOT / "artifacts" / "analysis" / "l5_cleanup_sanity"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# ===========================
# STEP 2: Label normalization helpers
# ===========================
def normalize_label_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip())


def dedupe_labels(labels: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in labels:
        n = normalize_label_text(x)
        if n and n not in seen:
            out.append(n)
            seen.add(n)
    return out


GLOBAL_CATEGORY_LIST = dedupe_labels(GLOBAL_CATEGORY_LIST)
print(f"Global category count: {len(GLOBAL_CATEGORY_LIST)}")


Global category count: 45


In [4]:
# ===========================
# STEP 3: Load source rows from Snowflake
# ===========================
def _quote_sql(s: str) -> str:
    return str(s).replace("'", "''")


all_target_labels = sorted({x for vals in TARGET_GROUPS.values() for x in vals})
labels_sql = ", ".join([f"'{_quote_sql(lbl)}'" for lbl in all_target_labels])

where_parts = [f"l.PROPOSED_LABEL IN ({labels_sql})"]
if EXCLUDE_INSERT_PRODUCTS:
    where_parts.append("UPPER(COALESCE(p.DESCRIPTION, '')) NOT LIKE '%INSERT%'")
if ONLY_PRICED:
    where_parts.append("UPPER(COALESCE(p.PRICING_STATUS_C, '')) = 'PRICED'")
where_sql = " AND ".join(where_parts)

query = f"""
SELECT
  p.*, 
  l.PROPOSED_LABEL,
  l.PROPOSED_CLUSTER
FROM {LABELS_TABLE} l
JOIN {PRODUCTS_TABLE} p
  ON p.{JOIN_ID_COL} = l.{JOIN_ID_COL}
WHERE {where_sql}
"""

sf = get_snowflake_session()
df = sf.sql(query).to_pandas()
df = df.drop_duplicates(subset=[JOIN_ID_COL]).reset_index(drop=True)

print(f"Rows loaded: {len(df):,}")
display(df["PROPOSED_LABEL"].value_counts().rename_axis("PROPOSED_LABEL").reset_index(name="ROW_COUNT"))


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZJbb6MwEIX%2FCvI%2Bg4GQklhJqly2alZtEzVppe2bC5PEAmzqMYH019fksuo%2BtFLfkDnH3%2FGcGVw3Re7sQaNQckgCzycOyESlQm6H5Gl94%2FaIg4bLlOdKwpAcAMn1aIC8yEs2rsxOPsJbBWgce5FE1v4YkkpLpjgKZJIXgMwkbDW%2Bv2Oh5zOOCNpYHDlbUhSWtTOmZJTWde3VHU%2FpLQ1936d%2Bn1pVK%2FlFPiHK7xmlVkYlKr9YGvumLxAB9aMWYRWWsDwbJ0KeRvAd5fUkQna7Xi%2Fd5WK1Js748rqpklgVoFeg9yKBp8e7UwC0Ccos6cZRt%2BdV6AJH4wYeSlVvcp5BooqyMvZaz37RDaQ0V1thhzWfDUmZibQ%2F6fQ6i9L8eXjdrg7V%2B37aSX9Plbl9WWV5EfXjxd9qg3GowjohzvOl2rCtdo5YwVy2hRp75IdXrh%2B5QbgOQxZFLPC9uNN9Ic7MFiokN0fnJTUmwg4JoEl2XG7BU5nhx5C8LOm%2F%2FBSazLyL5q1o7mOIs3Ry1Y8poqJtb%2BS0OuwYRI9%2BPJAB%2FWw%2Fr%2BGDbWY%2BW6pcJAfnRumCm6%2BLC7zgeCJSd3OUMii4yMdpqgHRFpjnqp5q4MZuu9EVEDo6Uf%2Ff99EH&RelayState=ver%3A3-hint%3A9376132675904534-ETMsDgAAAZ2D3eVvABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEFGOdqHrn8l8RZgDso3F1mYAAACgcCJ

,PROPOSED_LABEL,ROW_COUNT
0,Lab Equipment,19120
1,Lab Consumables,13774
2,General Lab Supplies,11778
3,General Lab Equipment,11303
4,Lab Instruments,11040
5,Lab Instruments and Tools,6787


In [5]:
# ===========================
# STEP 4: Preview each target group
# ===========================
# Show table/dataframe for each group (first pass requested)
group_frames = {}
for group_name, labels in TARGET_GROUPS.items():
    gdf = df[df["PROPOSED_LABEL"].isin(labels)].copy().reset_index(drop=True)
    group_frames[group_name] = gdf
    print(f"\n=== {group_name} ===")
    print(f"Rows: {len(gdf):,}")
    display(gdf[["PRODUCT_ID", "PRODUCT_NAME", "PROPOSED_LABEL"]].head(20))
    display(gdf["PROPOSED_LABEL"].value_counts().rename_axis("PROPOSED_LABEL").reset_index(name="ROW_COUNT"))



=== group_1_general_supplies ===
Rows: 25,552


,PRODUCT_ID,PRODUCT_NAME,PROPOSED_LABEL
0,01tPK00000GcUOXYA3,TOP REMOVABLE STAINLESS 11inch W,Lab Consumables
1,01tPK00000ID4FTYA1,HALEON SENSODYNE PRONAMEL TOOTHPASTE,Lab Consumables
2,01tPP00000DJmpjYAD,PerkinElmer Initial Calibration Verification S...,General Lab Supplies
3,01tPK00000ICwMPYA1,CUSTOM GEL TRAY FOR A UNITS,Lab Consumables
4,01tPK00000ID7vxYAD,ESSITY TORK HAND TOWEL,Lab Consumables
5,01tPP00000DkFUgYAN,"Glass Support Base, Fritted",Lab Consumables
6,01tPK00000GcKXSYA3,GMAX BEDPANS,Lab Consumables
7,01tPK00000GbhvcYAB,DUKAL ABD PADS,Lab Consumables
8,01tPK00000GcboTYAR,PC215 pH/ion/cond meter pw kit,General Lab Supplies
9,01tPK00000ICmsmYAD,INSULATION FOAM 2,Lab Consumables


,PROPOSED_LABEL,ROW_COUNT
0,Lab Consumables,13774
1,General Lab Supplies,11778



=== group_2_equipment_instruments ===
Rows: 48,250


,PRODUCT_ID,PRODUCT_NAME,PROPOSED_LABEL
0,01tPK00000ICuylYAD,Pi-per (Pi Only Mote),Lab Equipment
1,01tPK00000HcYGKYA3,Piston w/seal & spring 50/100µL for pipettor E...,General Lab Equipment
2,01tPP00000DJa7kYAD,"Cytiva Ashless Clippings, 500 g",Lab Instruments
3,01tPK00000ICszJYAT,"X4T TX1000CC PKG, IVD-MD 230V",Lab Equipment
4,01tPK00000ID6X1YAL,HORIBA DRUG SCREEN REAGENT,Lab Instruments
5,01tPK00000GcFc3YAF,"SYRINGE,1002.5 C-XP",Lab Equipment
6,01tPK00000ICWhBYAX,ASCENSIA DIASTIX REAGENT STRIPS FOR URINALYSIS,Lab Instruments
7,01tPK00000GcefzYAB,PM Package Tacta 10ul,Lab Equipment
8,01tPK00000ID6WbYAL,HORIBA SPECIAL CHEM,Lab Instruments
9,01tPK00000ID6cSYAT,HORIBA YUMIZEN 1200,Lab Equipment


,PROPOSED_LABEL,ROW_COUNT
0,Lab Equipment,19120
1,General Lab Equipment,11303
2,Lab Instruments,11040
3,Lab Instruments and Tools,6787


In [6]:
# ===========================
# STEP 5: LLM + parser helpers
# ===========================
client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)


def _payload_text(payload: dict) -> str:
    parts = payload.get("content", [])
    chunks = []
    for p in parts:
        if isinstance(p, dict) and p.get("type") == "text":
            chunks.append(str(p.get("text", "")))
    if chunks:
        return "\n".join(chunks).strip()
    return str(payload)


def _extract_json_obj(text: str):
    import ast

    raw = str(text or "").strip()
    if not raw:
        raise ValueError("Empty LLM response text")

    raw = re.sub(r"^```(?:json)?", "", raw.strip())
    raw = re.sub(r"```$", "", raw.strip())

    try:
        return json.loads(raw)
    except Exception:
        pass

    start = raw.find("{")
    end = raw.rfind("}")
    if start >= 0 and end > start:
        cand = raw[start : end + 1]
        try:
            return json.loads(cand)
        except Exception:
            try:
                obj = ast.literal_eval(cand)
                if isinstance(obj, (dict, list)):
                    return obj
            except Exception:
                pass

    # Last-ditch: try list slice
    lstart = raw.find("[")
    lend = raw.rfind("]")
    if lstart >= 0 and lend > lstart:
        lcand = raw[lstart : lend + 1]
        try:
            return json.loads(lcand)
        except Exception:
            try:
                return ast.literal_eval(lcand)
            except Exception:
                pass

    raise ValueError("Could not parse JSON/list from LLM output")


def _normalize_rows_payload(parsed) -> list[dict]:
    if isinstance(parsed, list):
        rows = parsed
    elif isinstance(parsed, dict):
        rows = parsed.get("rows")
        if rows is None:
            for k in ("results", "items", "data", "predictions"):
                if k in parsed:
                    rows = parsed[k]
                    break
    else:
        rows = None

    if rows is None:
        return []
    if isinstance(rows, dict):
        rows = [rows]
    if not isinstance(rows, list):
        return []
    return [r for r in rows if isinstance(r, dict)]


def _discover_available_anthropic_models(region: str) -> list[str]:
    import boto3

    ctl = boto3.client("bedrock", region_name=region)
    resp = ctl.list_foundation_models(byProvider="Anthropic", byOutputModality="TEXT")
    out = []
    for m in resp.get("modelSummaries", []):
        mid = m.get("modelId")
        if not mid:
            continue
        inf = set(m.get("inferenceTypesSupported", []))
        if inf and "ON_DEMAND" not in inf:
            continue
        out.append(mid)
    return sorted(set(out))


_DISCOVERED_MODELS_CACHE = None
_DISCOVERED_MODELS_PRINTED = False


def invoke_llm_with_fallback(body: dict) -> tuple[dict, str]:
    global _DISCOVERED_MODELS_CACHE, _DISCOVERED_MODELS_PRINTED
    last_err = None
    tried = []
    candidate_ids = list(LLM_MODEL_IDS)

    if ENABLE_MODEL_DISCOVERY:
        try:
            if _DISCOVERED_MODELS_CACHE is None:
                _DISCOVERED_MODELS_CACHE = _discover_available_anthropic_models(AWS_REGION)
            discovered = _DISCOVERED_MODELS_CACHE
            for m in discovered:
                if m not in candidate_ids:
                    candidate_ids.append(m)
            if discovered and not _DISCOVERED_MODELS_PRINTED:
                print(f"Discovered Anthropic models ({AWS_REGION}): {discovered[:6]}{' ...' if len(discovered) > 6 else ''}")
                _DISCOVERED_MODELS_PRINTED = True
        except Exception as e:
            if not _DISCOVERED_MODELS_PRINTED:
                print(f"Model discovery warning: {e}")
                _DISCOVERED_MODELS_PRINTED = True

    for model_id in candidate_ids:
        tried.append(model_id)
        try:
            resp = client.invoke_model(
                modelId=model_id,
                contentType="application/json",
                accept="application/json",
                body=json.dumps(body),
            )
            payload = json.loads(resp["body"].read().decode("utf-8"))
            return payload, model_id
        except Exception as e:
            last_err = e
            msg = str(e)
            if ("ResourceNotFound" in msg) or ("ValidationException" in msg) or ("AccessDenied" in msg) or ("Unsupported" in msg):
                continue
            raise

    raise RuntimeError(
        "No configured/discovered Anthropic model is available in this account/region. "
        f"Tried: {tried}. Last error: {last_err}"
    )


In [7]:
# ===========================
# STEP 6: Sanity-check function
# ===========================
def llm_sanity_check_labels(
    in_df: pd.DataFrame,
    candidate_labels: list[str],
    batch_size: int = 50,
) -> pd.DataFrame:
    required_cols = {"PRODUCT_ID", "PRODUCT_NAME", "PROPOSED_LABEL"}
    missing = sorted(required_cols - set(in_df.columns))
    if missing:
        raise ValueError(f"Missing required columns for sanity check: {missing}")

    allowed_labels = dedupe_labels(candidate_labels)
    allowed_set = set(allowed_labels)

    out = in_df.copy().reset_index(drop=True)
    out["PROPOSED_LABEL"] = out["PROPOSED_LABEL"].map(normalize_label_text)
    out["LLM_DECISION"] = None
    out["LLM_LABEL"] = None
    out["LLM_REASON"] = None
    out["LLM_CONFIDENCE"] = np.nan

    first_parse_debug_printed = False
    pending_ranges = [(start, min(start + batch_size, len(out))) for start in range(0, len(out), batch_size)]
    min_split_size = 12

    while pending_ranges:
        start, stop = pending_ranges.pop(0)
        chunk = out.iloc[start:stop].copy()
        curr_size = len(chunk)
        if curr_size == 0:
            continue
        items = [
            {
                "row_id": int(i),
                "product_name": str(r["PRODUCT_NAME"]),
                "proposed_label": normalize_label_text(r["PROPOSED_LABEL"]),
            }
            for i, r in chunk.iterrows()
        ]

        prompt = {
            "context": SANITY_GOAL_CONTEXT,
            "task": (
                "Given product_name and proposed_label, decide KEEP, REASSIGN, or REVIEW. "
                "REASSIGN must use one candidate label. REVIEW if uncertain. "
                "Return STRICT JSON only, no markdown, no prose."
            ),
            "candidate_labels": allowed_labels,
            "allow_keep": ALLOW_KEEP_DECISION,
            "rules": [
                "decision must be KEEP, REASSIGN, or REVIEW",
                "llm_label must be one of candidate_labels or REVIEW",
                "if decision=KEEP then llm_label equals proposed_label",
                "if allow_keep is false, do not return KEEP",
                "if decision=REVIEW then llm_label=REVIEW",
                "confidence between 0 and 1",
            ],
            "items": items,
            "output_format": {
                "rows": [
                    {
                        "row_id": 1,
                        "decision": "KEEP|REASSIGN|REVIEW",
                        "llm_label": "label_or_REVIEW",
                        "confidence": 0.0,
                    }
                ]
            },
        }

        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": int(min(3800, max(1200, 70 * curr_size))),
            "temperature": LLM_TEMPERATURE,
            "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(prompt)}]}],
        }

        parsed = {"rows": []}
        raw_text = ""
        for attempt in range(LLM_PARSE_RETRIES):
            try:
                payload, _model_id_used = invoke_llm_with_fallback(body)
                raw_text = _payload_text(payload)
                parsed = _extract_json_obj(raw_text)
                break
            except Exception:
                repair_prompt = {
                    "task": "Repair this response to valid JSON only. Output JSON only.",
                    "raw_text": raw_text if 'raw_text' in locals() else "",
                    "required_format": {
                        "rows": [{"row_id": 1, "decision": "KEEP|REASSIGN|REVIEW", "llm_label": "label_or_REVIEW", "confidence": 0.0}]
                    },
                }
                repair_body = {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 1800,
                    "temperature": 0,
                    "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(repair_prompt)}]}],
                }
                try:
                    repair_payload, _ = invoke_llm_with_fallback(repair_body)
                    parsed = _extract_json_obj(_payload_text(repair_payload))
                    break
                except Exception:
                    parsed = {"rows": []}
        rows = _normalize_rows_payload(parsed)
        if not rows:
            if (not first_parse_debug_printed) and raw_text:
                print("Debug raw LLM text preview (first failure):")
                print(raw_text[:600])
                first_parse_debug_printed = True
            if curr_size > min_split_size:
                mid = start + (curr_size // 2)
                pending_ranges.insert(0, (mid, stop))
                pending_ranges.insert(0, (start, mid))
                print(f"Warning: JSON parse failed for rows {start}:{stop}; retrying as smaller batches.")
                continue
            print(f"Warning: JSON parse failed for rows {start}:{stop}; unresolved rows default to REVIEW.")


        for row in rows:
            try:
                ridx = int(row.get("row_id"))
            except Exception:
                continue
            if ridx not in out.index:
                continue

            proposed = out.at[ridx, "PROPOSED_LABEL"]
            decision = str(row.get("decision", "REVIEW")).upper().strip()
            if decision not in {"KEEP", "REASSIGN", "REVIEW"}:
                decision = "REVIEW"
            if (not ALLOW_KEEP_DECISION) and decision == "KEEP":
                decision = "REASSIGN"

            llm_label = normalize_label_text(row.get("llm_label", "REVIEW"))
            if decision == "KEEP":
                llm_label = proposed if proposed in allowed_set else "REVIEW"
            elif decision == "REVIEW":
                llm_label = "REVIEW"
            elif llm_label not in allowed_set:
                decision = "REVIEW"
                llm_label = "REVIEW"
            if (not ALLOW_KEEP_DECISION) and decision == "REASSIGN" and llm_label == proposed:
                decision = "REVIEW"
                llm_label = "REVIEW"

            reason = ""
            try:
                conf = float(row.get("confidence", np.nan))
                if np.isfinite(conf):
                    conf = min(1.0, max(0.0, conf))
                else:
                    conf = np.nan
            except Exception:
                conf = np.nan

            out.at[ridx, "LLM_DECISION"] = decision
            out.at[ridx, "LLM_LABEL"] = llm_label
            out.at[ridx, "LLM_REASON"] = reason
            out.at[ridx, "LLM_CONFIDENCE"] = conf

    out["LLM_DECISION"] = out["LLM_DECISION"].fillna("REVIEW")
    out["LLM_LABEL"] = out["LLM_LABEL"].fillna("REVIEW")
    out["LLM_REASON"] = out["LLM_REASON"].fillna("")
    out["LLM_MATCH"] = out["LLM_LABEL"].eq(out["PROPOSED_LABEL"])
    return out


In [8]:
# ===========================
# STEP 7: Run cleanup for each group
# ===========================
# Run cleanup for each group and display outputs
cleanup_outputs = {}

for group_name, labels in TARGET_GROUPS.items():
    gdf = group_frames[group_name][["PRODUCT_ID", "PRODUCT_NAME", "PROPOSED_LABEL"]].copy()
    print(f"\n=== Running cleanup for {group_name} ({len(gdf):,} rows) ===")
    gout = llm_sanity_check_labels(
        gdf,
        candidate_labels=GLOBAL_CATEGORY_LIST,
        batch_size=SANITY_BATCH_SIZE,
    )
    cleanup_outputs[group_name] = gout

    display(
        gout["LLM_DECISION"]
        .value_counts(dropna=False)
        .rename_axis("LLM_DECISION")
        .reset_index(name="ROW_COUNT")
    )
    display(
        gout.groupby(["PROPOSED_LABEL", "LLM_LABEL"], dropna=False)
        .size()
        .rename("ROW_COUNT")
        .reset_index()
        .sort_values("ROW_COUNT", ascending=False)
        .head(50)
    )
    display(
        gout[[
            "PRODUCT_ID",
            "PRODUCT_NAME",
            "PROPOSED_LABEL",
            "LLM_DECISION",
            "LLM_LABEL",
            "LLM_CONFIDENCE",
            "LLM_REASON",
        ]].head(30)
    )



=== Running cleanup for group_1_general_supplies (25,552 rows) ===
Debug raw LLM text preview (first failure):
{
  "rows": [
    {
      "row_id": 0,
      "decision": "REASSIGN",
      "llm_label": "Laboratory Furniture",
      "reason": "The product appears to be a piece of laboratory furniture, so it should be reassigned to the 'Laboratory Furniture' category.",
      "confidence": 0.8
    },
    {
      "row_id": 1,
      "decision": "REVIEW",
      "llm_label": "REVIEW",
      "reason": "The product name 'HALEON SENSODYNE PRONAMEL TOOTHPASTE' does not appear to be a scientific or laboratory product, so it should be reviewed.",
      "confidence": 0.7
    },
    {
      "row_id": 2,
      "decision


KeyboardInterrupt: 

In [ ]:
# Meeting summary: compact scorecard + top reassignment paths
summary_rows = []
transition_rows = []

for group_name, gout in cleanup_outputs.items():
    total = len(gout)
    if total == 0:
        continue

    decision_counts = gout["LLM_DECISION"].value_counts(dropna=False)
    keep_n = int(decision_counts.get("KEEP", 0))
    reassign_n = int(decision_counts.get("REASSIGN", 0))
    review_n = int(decision_counts.get("REVIEW", 0))

    summary_rows.append({
        "group": group_name,
        "rows": total,
        "keep_count": keep_n,
        "keep_pct": round(100.0 * keep_n / total, 2),
        "reassign_count": reassign_n,
        "reassign_pct": round(100.0 * reassign_n / total, 2),
        "review_count": review_n,
        "review_pct": round(100.0 * review_n / total, 2),
    })

    reassignment_paths = (
        gout.loc[gout["LLM_DECISION"] == "REASSIGN", ["PROPOSED_LABEL", "LLM_LABEL"]]
        .groupby(["PROPOSED_LABEL", "LLM_LABEL"], dropna=False)
        .size()
        .rename("ROW_COUNT")
        .reset_index()
        .sort_values("ROW_COUNT", ascending=False)
        .head(10)
    )
    if len(reassignment_paths) > 0:
        reassignment_paths.insert(0, "group", group_name)
        transition_rows.append(reassignment_paths)

summary_df = pd.DataFrame(summary_rows).sort_values("group")
print("Meeting scorecard")
display(summary_df)

print("Top reassignment paths by group (top 10 each)")
if transition_rows:
    meeting_transitions_df = pd.concat(transition_rows, ignore_index=True)
    display(meeting_transitions_df)
else:
    meeting_transitions_df = pd.DataFrame(columns=["group", "PROPOSED_LABEL", "LLM_LABEL", "ROW_COUNT"])
    print("No REASSIGN rows found.")

Meeting scorecard


,group,rows,keep_count,keep_pct,reassign_count,reassign_pct,review_count,review_pct
0,group_1_general_supplies,25552,1449,5.67,16757,65.58,7346,28.75
1,group_2_equipment_instruments,48250,3187,6.61,39193,81.23,5870,12.17


Top reassignment paths by group (top 10 each)


,group,PROPOSED_LABEL,LLM_LABEL,ROW_COUNT
0,group_1_general_supplies,Lab Consumables,Lab Accessories,3129
1,group_1_general_supplies,General Lab Supplies,Laboratory Instruments,2952
2,group_1_general_supplies,General Lab Supplies,Laboratory Equipment,2720
3,group_1_general_supplies,General Lab Supplies,Analytical Balances,1460
4,group_1_general_supplies,Lab Consumables,Cleaning and Sterilization,998
5,group_1_general_supplies,General Lab Supplies,Lab Accessories,962
6,group_1_general_supplies,Lab Consumables,Protective Wear,585
7,group_1_general_supplies,General Lab Supplies,Medical Accessories,394
8,group_1_general_supplies,Lab Consumables,Labeling and Tape,383
9,group_1_general_supplies,Lab Consumables,Medical Accessories,259


In [ ]:
# ===========================
# STEP 8: Export CSV files
# ===========================
# Optional export: separate CSV per group
if SAVE_OUTPUT_FILES:
    for group_name, gout in cleanup_outputs.items():
        out_path = OUT_DIR / f"{group_name}_llm_cleanup.csv"
        gout.to_csv(out_path, index=False)
        print("Saved:", out_path)
else:
    print("SAVE_OUTPUT_FILES=False: showing results in notebook only (no CSV written).")


SAVE_OUTPUT_FILES=False: showing results in notebook only (no CSV written).
